# SDE Mapper Research Toolkit

In this notebook we will demonstrate a collection of methods we've researched in an attempt to automate or semi-automate the mapping of VLMD (variable level metadata) objects to SDEs (standard data elements). 

## Method 1 (LLM + Embeddings + Similarity Search)

### Set Up and Import Packages & Data

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from tqdm import trange


from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest
from vllm.sampling_params import GuidedDecodingParams

from pydantic import BaseModel

from sentence_transformers import SentenceTransformer
from torch.nn.functional import cosine_similarity

In [ ]:
VLMD_DATA_PATH = "HEAL_CDE_Mappings-HDP00895-FILTERED.csv"

SDE_DICTIONARY = "master_sde_v2.jsonl"

PROMPT_FORMAT_FILE = "sde_mapper_training_prompt.txt"


VLMD_BASE_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
VLMD_LORA_MODEL_PATH = "finetuned_lora_model_path"
VLMD_MODEL_TEMPERATURE = 0.0
VLMD_MODEL_MAX_TOKENS = 4096
BATCH_SIZE = 64


SIMILARITY_SEARCH_EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
SDE_PARENT = "HEAL"
SDE_SEARCH_TOP_K = 10
TOPIC_SEARCH_TOP_K = 3



In [ ]:
benchmark_df = pd.read_csv(VLMD_DATA_PATH)
benchmark_df["vlmd_object_index"] = list(range(0, len(benchmark_df)))
benchmark_df['research_topic'] = 'Clinical Research in Pain Management'

names, descriptions, parsed_sdes, sde_source, sde_topic, sde_index = [], [], [], [], [], []
index = 0
with open(SDE_DICTIONARY, 'r') as f:
    for line in f:
        try:
            sde = json.loads(line.strip())
            names.append(sde['sde_name'])
            descriptions.append(sde['sde_description'])
            parsed_sdes.append(f"{sde['sde_name']}: {sde['sde_description']}")
            sde_source.append(sde['sde_parent_repository'])
            sde_topic.append(sde['sde_parent_domain'])
            sde_index.append(index)
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON on line: {line.strip()} - {e}")
        index += 1

sde_dictionary = pd.DataFrame({'name': names, 'description': descriptions, 'sde_index': sde_index, 'sde_source': sde_source, 'sde_topic': sde_topic, 'parsed_sde': parsed_sdes})
sde_dictionary = sde_dictionary[sde_dictionary.sde_source == SDE_PARENT]

print(f"Length of sde_dictionary = {len(sde_dictionary)}")


In [ ]:
benchmark_df.head()

### Parse Data For Inference

Now we will parse through the benchmark VLMD and create prompts for LLM inference.

In [ ]:
with open(PROMPT_FORMAT_FILE, 'r', encoding='utf-8') as file:
    prompt_format = file.read()

prompts = []
for i in range(len(benchmark_df)):

    input = {"name": benchmark_df['field_title'].iloc[i], 
             "definition": benchmark_df['field_description'].iloc[i]}

    formated_prompt_instance = prompt_format.replace('<<input_vlmd>>', str(input))
    prompts.append(str(formated_prompt_instance))

inference_df = pd.DataFrame({"vlmd_object_index": benchmark_df.vlmd_object_index,
                             "research_topic": benchmark_df.research_topic,
                             "target_sde_name": benchmark_df.element_title, 
                             "target_sde_description": benchmark_df.element_description,
                             "source_vlmd_name": benchmark_df.field_title, 
                             "source_vlmd_description": benchmark_df.field_description,
                             "input_text": prompts})

inference_df.head()

### Generate SDEs

We next generate a prelimiary SDE or each VLMD object using a finetuned LLM.

In [ ]:
def vlmd_generated_sdes(df, llm, VLMD_LORA_MODEL_PATH, sampling_params, BATCH_SIZE):
    
    N = len(df)
    results_df = pd.DataFrame(columns=["study_record", "input_text", "generated_output_text"])

    for lo in trange(0, N, BATCH_SIZE):

        hi = lo + BATCH_SIZE
        if hi > N:
            hi = N

        temp_df_for_batch = pd.DataFrame(columns=["study_record", "vlmd_object_id", "input_text", "generated_output_text"])

        batch_labels = df.iloc[lo:hi].get("study_record")
        batch_object_indices = df.iloc[lo:hi].get("vlmd_object_index")
        batch_inputs = df.iloc[lo:hi]["input_text"]

        batch_outputs = llm.generate(
            batch_inputs,
            sampling_params,
            lora_request=LoRARequest("arpa-h-eval", 1, VLMD_LORA_MODEL_PATH),
        )
        batch_outputs = [o.outputs[0].text for o in batch_outputs]

        temp_df_for_batch["study_record"] = batch_labels
        temp_df_for_batch["vlmd_object_index"] = batch_object_indices
        temp_df_for_batch["input_text"] = batch_inputs
        temp_df_for_batch["generated_output_text"] = batch_outputs

        results_df = pd.concat([results_df, temp_df_for_batch], axis=0)

    return results_df


class ResponseSchema(BaseModel):
    sde_name: str
    sde_description: str

ref_schema = ResponseSchema.model_json_schema()
decoding_params = GuidedDecodingParams(json=ref_schema)

sampling_params = SamplingParams(
    temperature=VLMD_MODEL_TEMPERATURE,
    max_tokens=VLMD_MODEL_MAX_TOKENS,
    guided_decoding=decoding_params,
)

llm = LLM(model=VLMD_BASE_MODEL, enable_lora=True)
results_df = vlmd_generated_sdes(inference_df, llm, VLMD_LORA_MODEL_PATH, sampling_params, BATCH_SIZE)

inference_df = inference_df.merge(results_df[["vlmd_object_index", "generated_output_text"]], on="vlmd_object_index", how="left")
del results_df

### Select Research Topic

For each VLMD obect we select the top 3 most likely research topics which could contain the correct SDE object. We will give SDEs from these research topics priority in our suggested mappings.

In [ ]:
def research_topic_similarity_search(df, sde_dictionary, SIMILARITY_SEARCH_EMBEDDING_MODEL, TOPIC_SEARCH_TOP_K, SDE_PARENT):

    model = SentenceTransformer(SIMILARITY_SEARCH_EMBEDDING_MODEL)

    outputs_embeddings, sde_embeddings = [], []
    for topic in df.research_topic.unique():
        embedding = model.encode(topic, convert_to_tensor=True)
        outputs_embeddings.append(embedding)

    for topic in sde_dictionary.sde_topic.unique():
        embedding = model.encode([str(topic)], convert_to_tensor=True)
        sde_embeddings.append(embedding)

    results = []
    for i in range(len(outputs_embeddings)):
        
        similarities = np.zeros(len(sde_embeddings))
        for j in range(len(sde_embeddings)):
            similarities[j] = cosine_similarity(outputs_embeddings[i], sde_embeddings[j])

        sorted_indices = np.argsort(similarities)[::-1]    
        top_k_similarities = similarities[sorted_indices[:TOPIC_SEARCH_TOP_K]]
        top_k_indices = sorted_indices[:TOPIC_SEARCH_TOP_K]
        
        results.append({
            "item_index_list1": df.research_topic.unique(),
            "top_k_matches": [
                {"item_index_list2": idx, "similarity_score": score}
                for idx, score in zip(top_k_indices, top_k_similarities)
            ]})

    return results


results = research_topic_similarity_search(inference_df, sde_dictionary, SIMILARITY_SEARCH_EMBEDDING_MODEL, TOPIC_SEARCH_TOP_K, SDE_PARENT)

topic_results = []
for result in results:
    selected_topics = []
    for match in result['top_k_matches']:
        topic_index = match['item_index_list2']
        topic = sde_dictionary.sde_topic.unique()[topic_index]
        selected_topics.append(topic)
    topic_results.append(selected_topics)

inference_df['top_research_topics'] = inference_df['research_topic'].map({'Clinical Research in Pain Management': topic_results})
del results

print(topic_results)

### Search For Recommended SDEs

For each VLMD object we use the respective, generated SDE to find the top 10 most similar SDEs from the SDE dictionary. Specifiically, we search only search the SDE data dictionary over the relevant SDE parent type (i.e. HEAL).

In [ ]:
model = SentenceTransformer(SIMILARITY_SEARCH_EMBEDDING_MODEL)

parsed_output = []
for i in range(len(inference_df)):
    try:
        sde = json.loads(inference_df.generated_output_text.iloc[i])
        parsed_output.append(f"{sde['sde_name']}: {sde['sde_description']}")     
    except:
        parsed_output.append("invalid: invalid")
            
inference_df["parsed_generated_sde_output"] = parsed_output

outputs_embeddings, sde_embeddings = [], []
for i in range(len(inference_df)):
    embedding = model.encode(inference_df.parsed_generated_sde_output.iloc[i], convert_to_tensor=True)
    outputs_embeddings.append(embedding)

for i in range(len(sde_dictionary)):
    embedding = model.encode([sde_dictionary.parsed_sde.iloc[i]], convert_to_tensor=True)
    sde_embeddings.append(embedding)

In [ ]:
results = []
for i in range(len(outputs_embeddings)):
    
    similarities = np.zeros(len(sde_embeddings))
    for j in range(len(sde_embeddings)):
        similarities[j] = cosine_similarity(outputs_embeddings[i], sde_embeddings[j])

    sorted_indices = np.argsort(similarities)[::-1]    
    top_k_similarities = similarities[sorted_indices[:SDE_SEARCH_TOP_K]]
    top_k_indices = sorted_indices[:SDE_SEARCH_TOP_K]
    
    results.append({
        "item_index_list1": i,
        "top_k_matches": [
            {"item_index_list2": idx, "similarity_score": score}
            for idx, score in zip(top_k_indices, top_k_similarities)
        ]
        })
    
results_df = pd.DataFrame({"vlmd_object_index": inference_df.vlmd_object_index, 
                            "parsed_generated_sde_output": inference_df["parsed_generated_sde_output"],
                            "suggested_sdes": results})

inference_df = inference_df.merge(results_df[["vlmd_object_index", "suggested_sdes"]], on="vlmd_object_index", how="left")
del results_df

### Organize Top SDEs

Now we return the top SDEs based on their similarity score, with preference given to those with the selected target research topics.

In [ ]:
sde_results = []
for i in range(len(inference_df)):
    suggested_sdes = inference_df.suggested_sdes.iloc[i] 
    selected_sdes = []
    on_topic_rank_indicator = []

    for match in suggested_sdes['top_k_matches']:

        sde_index = match['item_index_list2']
        sde = sde_dictionary.parsed_sde.iloc[sde_index]
        selected_sdes.append({sde: match['similarity_score']})
        topic = sde_dictionary.sde_topic.iloc[sde_index]
        if topic in inference_df.top_research_topics.iloc[i][0]:
            on_topic_rank_indicator.append(inference_df.top_research_topics.iloc[i][0].index(topic))
        else:
            on_topic_rank_indicator.append(TOPIC_SEARCH_TOP_K + 1)

    ordered_sdes = [x for _, x in sorted(enumerate(selected_sdes), key=lambda y: on_topic_rank_indicator[y[0]], reverse=False)]
    sde_results.append(ordered_sdes)

inference_df["ranked_sde_suggestions"] = sde_results

inference_df.head()

### Evaluate

We evaluate the performance of our model and see if the method recomends the correct SDE. If the correct SDE is recommened, then we check how highly ranked was the correct recommendation. 

In [ ]:
location_rank = []
no_query_index = []
for i in range(len(inference_df)):

    source_vlmd_name = inference_df.source_vlmd_name.iloc[i]
    source_vlmd_description = inference_df.source_vlmd_description.iloc[i]

    target_sde_name = inference_df.target_sde_name.iloc[i]
    target_sde_description = inference_df.target_sde_description.iloc[i]  

    parsed_truth = f"{target_sde_name}: {target_sde_description}"
    ranked_sdes = inference_df.ranked_sde_suggestions.iloc[i]
    parsed_ranked_sdes = [list(x.keys())[0].replace('\n', ' ').replace('\\n', ' ') for x in ranked_sdes]

    if parsed_truth in parsed_ranked_sdes:
        location_rank.append(parsed_ranked_sdes.index(parsed_truth) + 1)
    else:
        location_rank.append(99)

inference_df['suggested_sde_eval_rank'] = location_rank 

for k in [1, 5, 10]:
    score = np.round(100*np.sum(inference_df.suggested_sde_eval_rank <= k) / len(inference_df), 1)
    print(f"Correct SDE identified within top {k} suggestions - {score}%")

## Method 2 (Embedding + Similarity Search)

### Set Up and Import Packages & Data

In [ ]:
import chromadb
import pandas as pd
import os
from FlagEmbedding import FlagModel
# set which GPU to use
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
target_file = "master_sde_v2.jsonl"

benchmark_file = "HEAL_CDE_Mappings-HDP00895-FILTERED.csv"
# base model "BAAI/bge-large-en-v1.5" or custom embedding model "uc-ctds/bge-large-en-v1.5-bio-mapping"
embedding_model = "uc-ctds/bge-large-en-v1.5-bio-mapping" 

target_df = pd.read_json(target_file, lines=True)
target_df.head()

In [ ]:
def format_name_type_and_desc(row):
    return f"{row['sde_name']} ({row['sde_data_type']}): {row['sde_description']}"

target_df["name_type_and_desc"] = target_df.apply(format_name_type_and_desc, axis=1)
target_df["name_type_and_desc"].iloc[0]


In [ ]:
target_df.head(n=3)

### Create Embeddings

In [ ]:
model = FlagModel(embedding_model, use_fp16=True)
models = {
    "name_type_and_desc": model
}

embeddings = {
    "name_type_and_desc": models["name_type_and_desc"].encode(target_df["name_type_and_desc"].tolist())
}

In [ ]:
client = chromadb.Client()

try:
    # delete collection if already exists
    client.delete_collection(name='prop_name_desc')
except Exception:
    print('collection does not exist, do nothing')

In [ ]:
collection = client.create_collection("prop_name_desc", metadata={"hnsw:space": "cosine"})


batch_size = 5000
for i in range(0, target_df.shape[0], batch_size):
    print(f'adding records to collection, from {i} to {i+batch_size}')
    # just supply a list of embeddings and metadata to chroma
    # see https://docs.trychroma.com/docs/collections/add-data
    collection.add(
        embeddings=embeddings["name_type_and_desc"][i:i+batch_size].tolist(),
        ids=[str(id) for id in target_df.index[i:i+batch_size].tolist()]
        #metadatas=target_df.to_dict("records")[i:i+batch_size]
    )


### Evaluation

In [ ]:
def get_top_k(query_combined_emb, k):
    results = collection.query(
        query_embeddings=query_combined_emb,
        n_results=k
    )
    return results


def return_top_k_results(row, k):
    format_name_type_and_desc = f"{row['field_name']} ({row['field_type']}): {row['field_description']}"
    query_combined_emb = models["name_type_and_desc"].encode(format_name_type_and_desc)
    results = get_top_k(query_combined_emb, k)
    return results


def index_to_name(index_list):
    name_list = []
    for index in index_list:
        name = target_df.loc[int(index)]["sde_name"]
        name_list.append(name)
    return name_list


def format_results(results):
    formatted_results_df = pd.DataFrame({
        'ids': results['ids'][0],
        'distances': results['distances'][0]
    })
    formatted_results_df.sort_values(by='distances', ascending=True, inplace=True)
    formatted_results_df['names'] = index_to_name(formatted_results_df['ids'])
    return pd.Series([
        formatted_results_df['ids'].to_list(),
        formatted_results_df['distances'].to_list(),
        formatted_results_df['names'].to_list()
    ])


def process_benchmark(benchmark_name):
    print(f'processing {benchmark_name}')
    df = pd.read_csv(benchmark_name)
    df = df.dropna(subset=["field_name", "field_description", "element_title", "element_description"])
    # embed query variables -- var name, var desc and return top_k
    # by searching CDE embeddings
    top_k_list = [1, 5, 10]
    for k in top_k_list:
        col_name = f'top_{k}_results'
        df[col_name] = df.apply(lambda x: return_top_k_results(x, k=k), axis=1)
        # format results
        output_cols = f'top_{k}_ids,top_{k}_distances,top_{k}_names'.split(',')
        df[output_cols] = df[col_name].apply(
            lambda x: format_results(x)
        )
    print('returning top k results')
    return df

def calculate_acc(truth, pred):
  correct = 0
  for t, p_list_of_names in zip(truth, pred):
      if t in p_list_of_names:
          correct += 1
  return correct / len(truth)


def run_evals(df, benchmark_name, embedding_model):
    # print('calculating metrics')
    evals = {}
    row_index = []
    top_k_list = [1, 5, 10]
    for k in top_k_list:
        evals[f'accuracy_{k}'] = []

    row_index.append(f'{benchmark_name}_{df.shape[0]}_{embedding_model}')
    for k in top_k_list:
        col_name = f'top_{k}_names'
        top_k_names_list = df[col_name].to_list()
        truth_list = df['element_title'].to_list()
        accuracy = calculate_acc(truth_list, top_k_names_list)
        evals[f'accuracy_{k}'].append(accuracy)

    # print('returning metrics')
    return pd.DataFrame(evals, index=row_index)

def run_evals_per_row(row, k):
    col_name = f'top_{k}_names'
    top_k_names_list = row[col_name]
    truth = row['element_title']
    is_match = truth in top_k_names_list
    return is_match


def get_metrics_per_row(df):
    top_k_list = [5]
    for k in top_k_list:
        output_cols = f'is_match_in_top_{k}_name_desc'
        df[output_cols] = df.apply(lambda x: run_evals_per_row(x, k), axis=1)
    return df

In [ ]:
benchmark_name = benchmark_file
df = process_benchmark(benchmark_name=benchmark_name)
metrics_per_row = get_metrics_per_row(df)

In [ ]:
metrics_per_row['is_match_in_top_5_name_desc'].value_counts()


In [ ]:
cols_to_keep = [
    'field_name', 'field_description', 'field_type',
    'element_title', 'element_description', 'top_5_results', 'top_5_ids', 'top_5_distances',
    'top_5_names',  'is_match_in_top_5_name_desc'
    ]

metrics_per_row.to_csv('method2_metrics_per_row.csv', sep='\t', columns=cols_to_keep)


In [ ]:
benchmark_names = [benchmark_file]
results = pd.concat([
    run_evals(df=process_benchmark(eval_data), benchmark_name=eval_data, embedding_model=embedding_model)
    for eval_data in benchmark_names
])

results